# Week 3 – Unsupervised Learning & Model Evaluation

Same workflow as `week3_analysis.py`, cell by cell, with outputs. **Part A** clusters the employees without using any labels (PCA, K-Means, hierarchical). **Part B** evaluates classifiers that predict Attrition (cross-validation, tuning, confusion matrix, precision/recall/F1, ROC-AUC).

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram, cophenet
from scipy.spatial.distance import pdist
from scipy.stats import chi2_contingency

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import (silhouette_score, silhouette_samples,
                             calinski_harabasz_score, davies_bouldin_score,
                             adjusted_rand_score, normalized_mutual_info_score,
                             accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, roc_curve, confusion_matrix,
                             precision_recall_curve)
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     RepeatedStratifiedKFold, GridSearchCV,
                                     cross_validate, cross_val_predict)
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier

warnings.filterwarnings("ignore")
for folder in ("figures", "results"):
    os.makedirs(folder, exist_ok=True)

SEED = 42
NUM_COLS = ["Age", "Experience_Years", "Monthly_Income",
            "Satisfaction_Score", "Weekly_Work_Hours"]
CAT_COLS = ["Education", "Department", "City", "Job_Level"]


def section(title):
    print("\n" + "=" * 64)
    print(title)
    print("=" * 64)

## 1. Load data

Load the data.

In [2]:
df = pd.read_csv("data/employee_data.csv")
print("shape:", df.shape, "| duplicates:", df.duplicated().sum())
print("missing:", df.isnull().sum()[df.isnull().sum() > 0].to_dict())

# ===============================================================
#  PART A - UNSUPERVISED LEARNING (Attrition is NOT used here)
# ===============================================================

shape: (500, 10) | duplicates: 0
missing: {'Age': 15, 'Monthly_Income': 18, 'Education': 10, 'Department': 8, 'Satisfaction_Score': 12}


## 2. Prepare the clustering features

Impute + standardize the five numeric features. Clustering uses distances, so features must be on the same scale; the check below shows what goes wrong without scaling.

In [3]:
prep = Pipeline([("impute", SimpleImputer(strategy="median")),
                 ("scale", StandardScaler())])
Xs = prep.fit_transform(df[NUM_COLS])           # no labels involved, so using all rows is fine
print("clustering on", NUM_COLS)
print("\ncorrelation between the clustering features:")
corr = df[NUM_COLS].corr().round(2)
print(corr.to_string())

# why scaling matters: income is ~100,000x bigger than satisfaction in raw units
raw = SimpleImputer(strategy="median").fit_transform(df[NUM_COLS])
lab_raw = KMeans(3, n_init=20, random_state=SEED).fit_predict(raw)
lab_std = KMeans(3, n_init=20, random_state=SEED).fit_predict(Xs)
income_only = KMeans(3, n_init=20, random_state=SEED).fit_predict(raw[:, [2]])
print("\nK-Means (K=3) agreement, adjusted Rand index:")
print("  raw features vs income alone:   ", round(adjusted_rand_score(lab_raw, income_only), 3))
print("  scaled features vs raw features:", round(adjusted_rand_score(lab_std, lab_raw), 3))
print("  -> on raw data the clusters are basically just income bands")

clustering on ['Age', 'Experience_Years', 'Monthly_Income', 'Satisfaction_Score', 'Weekly_Work_Hours']

correlation between the clustering features:
                     Age  Experience_Years  Monthly_Income  Satisfaction_Score  Weekly_Work_Hours
Age                 1.00              0.91            0.86                0.04               0.01
Experience_Years    0.91              1.00            0.93                0.01               0.04
Monthly_Income      0.86              0.93            1.00               -0.00               0.11
Satisfaction_Score  0.04              0.01           -0.00                1.00              -0.04
Weekly_Work_Hours   0.01              0.04            0.11               -0.04               1.00

K-Means (K=3) agreement, adjusted Rand index:
  raw features vs income alone:    1.0
  scaled features vs raw features: 0.542
  -> on raw data the clusters are basically just income bands


## 3. PCA (dimensionality reduction)

PCA rotates the data into new axes ordered by how much variance they capture. Used here to understand the features and to draw the clusters in 2D.

In [4]:
pca_full = PCA(random_state=SEED).fit(Xs)
evr = pca_full.explained_variance_ratio_
print("explained variance per component:", evr.round(3).tolist())
print("cumulative:", np.cumsum(evr).round(3).tolist())
loadings = pd.DataFrame(pca_full.components_[:2].T, index=NUM_COLS,
                        columns=["PC1", "PC2"]).round(2)
print("\nloadings (how each feature contributes):")
print(loadings.to_string())
pca2 = PCA(n_components=2, random_state=SEED)
Xp = pca2.fit_transform(Xs)
loadings.to_csv("results/pca_loadings.csv")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].bar(range(1, 6), evr, color="#4C72B0", label="per component")
ax[0].plot(range(1, 6), np.cumsum(evr), "o-", color="#C44E52", label="cumulative")
ax[0].set_xlabel("principal component"); ax[0].set_ylabel("explained variance")
ax[0].set_title("PCA scree plot"); ax[0].legend()
im = ax[1].imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax[1].set_xticks(range(5)); ax[1].set_yticks(range(5))
ax[1].set_xticklabels(NUM_COLS, rotation=40, ha="right"); ax[1].set_yticklabels(NUM_COLS)
for i in range(5):
    for j in range(5):
        ax[1].text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
ax[1].set_title("Correlation of clustering features")
plt.tight_layout()
plt.savefig("figures/01_pca_and_correlation.png", dpi=110)
plt.close()

explained variance per component: [0.553, 0.208, 0.192, 0.034, 0.013]
cumulative: [0.553, 0.762, 0.953, 0.987, 1.0]

loadings (how each feature contributes):
                     PC1   PC2
Age                 0.57  0.07
Experience_Years    0.59  0.02
Monthly_Income      0.57 -0.04
Satisfaction_Score  0.01  0.71
Weekly_Work_Hours   0.05 -0.70


![01_pca_and_correlation.png](figures/01_pca_and_correlation.png)

## 4. K-Means: choosing K

Try K = 2 to 8 and judge each with inertia (elbow), silhouette, Calinski-Harabasz, Davies-Bouldin and a stability check.

In [5]:
rows = []
for k in range(2, 9):
    km = KMeans(k, n_init=20, random_state=SEED).fit(Xs)
    rows.append({"K": k, "Inertia": km.inertia_,
                 "Silhouette": silhouette_score(Xs, km.labels_),
                 "Calinski_Harabasz": calinski_harabasz_score(Xs, km.labels_),
                 "Davies_Bouldin": davies_bouldin_score(Xs, km.labels_)})
kres = pd.DataFrame(rows)

# stability: fit on random 80% subsets, label everyone, compare runs with ARI
rng = np.random.RandomState(SEED)
stab = []
for k in range(2, 9):
    runs = []
    for r in range(10):
        idx = rng.choice(len(Xs), int(0.8 * len(Xs)), replace=False)
        runs.append(KMeans(k, n_init=5, random_state=r).fit(Xs[idx]).predict(Xs))
    aris = [adjusted_rand_score(runs[i], runs[j])
            for i in range(10) for j in range(i + 1, 10)]
    stab.append(np.mean(aris))
kres["Stability_ARI"] = stab
print(kres.round(3).to_string(index=False))
kres.round(4).to_csv("results/kmeans_evaluation.csv", index=False)
best_k = int(kres.loc[kres.Silhouette.idxmax(), "K"])
print("\nK with the highest silhouette:", best_k)

fig, ax = plt.subplots(2, 2, figsize=(10, 7))
for a, col, title in zip(ax.ravel(),
                         ["Inertia", "Silhouette", "Calinski_Harabasz", "Davies_Bouldin"],
                         ["Inertia (elbow plot, lower = tighter)", "Silhouette (higher = better)",
                          "Calinski-Harabasz (higher = better)", "Davies-Bouldin (lower = better)"]):
    a.plot(kres.K, kres[col], "o-")
    a.set_title(title, fontsize=10); a.set_xlabel("K")
plt.tight_layout()
plt.savefig("figures/02_kmeans_k_selection.png", dpi=110)
plt.close()

 K  Inertia  Silhouette  Calinski_Harabasz  Davies_Bouldin  Stability_ARI
 2 1546.316       0.318            307.139           1.196          0.954
 3 1292.988       0.232            231.976           1.429          0.787
 4 1122.436       0.215            202.913           1.448          0.716
 5  979.189       0.222            192.200           1.318          0.788
 6  885.470       0.216            180.148           1.277          0.654
 7  819.727       0.209            168.425           1.268          0.554
 8  765.103       0.208            159.375           1.236          0.485

K with the highest silhouette: 2


![02_kmeans_k_selection.png](figures/02_kmeans_k_selection.png)

## 5. Silhouette profile for the chosen K

Silhouette of every point in the chosen clustering: how well it fits its own cluster compared with the nearest other one.

In [6]:
km = KMeans(best_k, n_init=20, random_state=SEED).fit(Xs)
km_labels = km.labels_
sil = silhouette_samples(Xs, km_labels)
for c in range(best_k):
    print(f"cluster {c}: size {np.sum(km_labels == c)}, mean silhouette {sil[km_labels == c].mean():.3f}, "
          f"share below 0: {np.mean(sil[km_labels == c] < 0):.1%}")
fig, ax = plt.subplots(figsize=(6.5, 4.5))
y0 = 0
for c in range(best_k):
    vals = np.sort(sil[km_labels == c])
    ax.fill_betweenx(np.arange(y0, y0 + len(vals)), 0, vals, alpha=0.75, label=f"cluster {c}")
    y0 += len(vals) + 10
ax.axvline(sil.mean(), color="red", ls="--", label=f"mean = {sil.mean():.2f}")
ax.set_xlabel("silhouette value"); ax.set_yticks([])
ax.set_title(f"Silhouette profile, K = {best_k}"); ax.legend()
plt.tight_layout()
plt.savefig("figures/03_silhouette_profile.png", dpi=110)
plt.close()

cluster 0: size 237, mean silhouette 0.291, share below 0: 1.3%
cluster 1: size 263, mean silhouette 0.342, share below 0: 0.0%


![03_silhouette_profile.png](figures/03_silhouette_profile.png)

## 6. Hierarchical clustering

Ward, complete and average linkage compared with silhouette and cophenetic correlation, plus a dendrogram. Also checks how many points end up in the smallest cluster.

In [7]:
hrows = []
dists = pdist(Xs)
for method in ["ward", "complete", "average"]:
    coph = cophenet(linkage(Xs, method=method), dists)[0]
    for k in range(2, 7):
        lab = AgglomerativeClustering(n_clusters=k, linkage=method).fit_predict(Xs)
        hrows.append({"Linkage": method, "K": k,
                      "Silhouette": silhouette_score(Xs, lab),
                      "Cophenetic": coph,
                      "Smallest_cluster": int(np.bincount(lab).min())})
hres = pd.DataFrame(hrows)
print(hres.round(3).to_string(index=False))
hres.round(4).to_csv("results/hierarchical_evaluation.csv", index=False)

ward_lab = AgglomerativeClustering(n_clusters=best_k, linkage="ward").fit_predict(Xs)
print(f"\nagreement (ARI) between K-Means and Ward at K={best_k}:",
      round(adjusted_rand_score(km_labels, ward_lab), 3))

fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
for method, mk in zip(["ward", "complete", "average"], ["o", "s", "^"]):
    s = hres[hres.Linkage == method]
    ax[0].plot(s.K, s.Silhouette, marker=mk, label=method)
ax[0].plot(kres.K[kres.K <= 6], kres.Silhouette[kres.K <= 6], "k--", label="k-means")
ax[0].set_xlabel("number of clusters"); ax[0].set_ylabel("silhouette")
ax[0].set_title("Silhouette: hierarchical linkages vs K-Means"); ax[0].legend()
Z = linkage(Xs, method="ward")
dendrogram(Z, truncate_mode="lastp", p=20, ax=ax[1], leaf_rotation=90, leaf_font_size=8,
           color_threshold=Z[-best_k + 1, 2] - 1e-6 if best_k > 1 else None)
ax[1].set_title("Ward dendrogram (last 20 merges)")
ax[1].set_ylabel("merge distance"); ax[1].set_xlabel("(number of points in branch)")
plt.tight_layout()
plt.savefig("figures/04_hierarchical.png", dpi=110)
plt.close()

 Linkage  K  Silhouette  Cophenetic  Smallest_cluster
    ward  2       0.296       0.519               193
    ward  3       0.192       0.519               140
    ward  4       0.157       0.519                63
    ward  5       0.156       0.519                63
    ward  6       0.161       0.519                50
complete  2       0.302       0.539               115
complete  3       0.169       0.539               115
complete  4       0.133       0.539                21
complete  5       0.139       0.539                21
complete  6       0.112       0.539                21
 average  2       0.356       0.634                16
 average  3       0.201       0.634                 1
 average  4       0.215       0.634                 1
 average  5       0.185       0.634                 1
 average  6       0.168       0.634                 1

agreement (ARI) between K-Means and Ward at K=2: 0.577


![04_hierarchical.png](figures/04_hierarchical.png)

## 7. Interpreting the clusters

What do the clusters mean? Profiles in original units, comparison with labels the clustering never saw, and a chi-square test on attrition.

In [8]:
out = df.copy()
out["Cluster"] = km_labels
prof = out.groupby("Cluster")[NUM_COLS].mean().round(1)
prof["Size"] = out.groupby("Cluster").size()
print("average value per cluster (original units):")
print(prof.to_string())
prof.to_csv("results/cluster_profiles.csv")

print("\njob level inside each cluster (row %):")
jl = pd.crosstab(out["Cluster"], out["Job_Level"], normalize="index").round(2)
print(jl.to_string())

print("\nattrition rate per cluster (Attrition was NOT used to build clusters):")
att = out.groupby("Cluster")["Attrition"].agg(["mean", "sum", "count"]).round(3)
att.columns = ["Attrition_rate", "Left", "Size"]
print(att.to_string())
chi2, pval, _, _ = chi2_contingency(pd.crosstab(out["Cluster"], out["Attrition"]))
print(f"chi-square test, cluster vs attrition: chi2 = {chi2:.2f}, p = {pval:.4f}")
att.to_csv("results/cluster_attrition.csv")

# external check against a label that exists but was not used (Job_Level)
print("\nexternal validation against Job_Level (a label the clustering never saw):")
print("  adjusted Rand index:", round(adjusted_rand_score(out["Job_Level"], km_labels), 3))
print("  normalized mutual info:", round(normalized_mutual_info_score(out["Job_Level"], km_labels), 3))
print("  (0 = no relation, 1 = perfect match)")

out["PC1"], out["PC2"] = Xp[:, 0], Xp[:, 1]
out.to_csv("results/cluster_assignments.csv", index=False)

fig, ax = plt.subplots(1, 3, figsize=(15, 4.3))
for c in range(best_k):
    m = out.Cluster == c
    ax[0].scatter(out.PC1[m], out.PC2[m], s=14, alpha=0.7, label=f"cluster {c}")
ax[0].set_title("K-Means clusters in PCA space"); ax[0].legend()
colors = {"Junior": "#4C72B0", "Mid": "#55A868", "Senior": "#DD8452", "Lead": "#C44E52"}
for lvl, col in colors.items():
    m = out.Job_Level == lvl
    ax[1].scatter(out.PC1[m], out.PC2[m], s=14, alpha=0.7, color=col, label=lvl)
ax[1].set_title("Same points coloured by Job_Level"); ax[1].legend()
for v, col, nm in [(0, "#4C72B0", "stayed"), (1, "#C44E52", "left")]:
    m = out.Attrition == v
    ax[2].scatter(out.PC1[m], out.PC2[m], s=14, alpha=0.5, color=col, label=nm)
ax[2].set_title("Same points coloured by Attrition"); ax[2].legend()
for a in ax:
    a.set_xlabel(f"PC1 ({evr[0]:.0%})"); a.set_ylabel(f"PC2 ({evr[1]:.0%})")
plt.tight_layout()
plt.savefig("figures/05_pca_clusters.png", dpi=110)
plt.close()


# ===============================================================
#  PART B - SUPERVISED MODEL EVALUATION (predict Attrition)
# ===============================================================

def make_prep(num_cols=NUM_COLS):
    num = Pipeline([("impute", SimpleImputer(strategy="median")),
                    ("scale", StandardScaler())])
    cat = Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])
    return ColumnTransformer([("num", num, num_cols), ("cat", cat, CAT_COLS)])


def pipe(model, with_pca=False, num_cols=NUM_COLS):
    steps = [("prep", make_prep(num_cols))]
    if with_pca:
        steps.append(("pca", PCA(random_state=SEED)))
    steps.append(("model", model))
    return Pipeline(steps)

average value per cluster (original units):
          Age  Experience_Years  Monthly_Income  Satisfaction_Score  Weekly_Work_Hours  Size
Cluster                                                                                     
0        35.6              14.2         93456.8                 6.5               42.1   237
1        25.0               3.2         43802.5                 6.5               41.9   263

job level inside each cluster (row %):
Job_Level  Junior  Lead   Mid  Senior
Cluster                              
0            0.33  0.08  0.39    0.21
1            0.34  0.10  0.39    0.17

attrition rate per cluster (Attrition was NOT used to build clusters):
         Attrition_rate  Left  Size
Cluster                            
0                 0.397    94   237
1                 0.270    71   263
chi-square test, cluster vs attrition: chi2 = 8.48, p = 0.0036

external validation against Job_Level (a label the clustering never saw):
  adjusted Rand index: -0.002
  normal

![05_pca_clusters.png](figures/05_pca_clusters.png)

## 8. Split

Stratified 80/20 split for the supervised part.

In [9]:
X = df.drop(columns=["Attrition"])
y = df["Attrition"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y)
print("train:", X_train.shape, "test:", X_test.shape,
      "| leavers: train", int(y_train.sum()), "test", int(y_test.sum()))

train: (400, 9) test: (100, 9) | leavers: train 132 test 33


## 9. Compare models with repeated cross-validation (train data only)

Seven models compared with 5-fold CV repeated 3 times, on the training data only.

In [10]:
cv_rep = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=SEED)
candidates = {
    "Baseline (always stayed)": pipe(DummyClassifier(strategy="most_frequent")),
    "Logistic Regression": pipe(LogisticRegression(max_iter=2000, class_weight="balanced")),
    "PCA + Logistic Regression": pipe(LogisticRegression(max_iter=2000, class_weight="balanced"), with_pca=True),
    "Decision Tree": pipe(DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=SEED)),
    "Random Forest": pipe(RandomForestClassifier(n_estimators=200, max_depth=4, min_samples_leaf=5,
                                                 class_weight="balanced", random_state=SEED)),
    "KNN": pipe(KNeighborsClassifier(n_neighbors=21)),
    "Gradient Boosting": pipe(HistGradientBoostingClassifier(max_depth=2, learning_rate=0.05,
                                                             max_iter=100, random_state=SEED)),
}
scoring = ["accuracy", "precision", "recall", "f1", "roc_auc"]
cv_rows = []
for name, model in candidates.items():
    r = cross_validate(model, X_train, y_train, cv=cv_rep, scoring=scoring)
    row = {"Model": name}
    for s in scoring:
        row[s] = r[f"test_{s}"].mean()
    row["roc_auc_std"] = r["test_roc_auc"].std()
    cv_rows.append(row)
cv_df = pd.DataFrame(cv_rows)
print(cv_df.round(3).to_string(index=False))
cv_df.round(4).to_csv("results/cross_validation_comparison.csv", index=False)

                    Model  accuracy  precision  recall    f1  roc_auc  roc_auc_std
 Baseline (always stayed)     0.670      0.000   0.000 0.000    0.500        0.000
      Logistic Regression     0.541      0.368   0.541 0.436    0.576        0.058
PCA + Logistic Regression     0.541      0.368   0.541 0.436    0.576        0.058
            Decision Tree     0.523      0.362   0.585 0.429    0.556        0.049
            Random Forest     0.571      0.372   0.411 0.388    0.580        0.049
                      KNN     0.668      0.394   0.058 0.100    0.548        0.065
        Gradient Boosting     0.665      0.493   0.154 0.228    0.556        0.051


## 10. Can extra features rescue the classifiers?

Do three extra features change anything?

In [11]:
def add_features(d):
    d = d.copy()
    d["Income_per_Exp_Year"] = d["Monthly_Income"] / (d["Experience_Years"] + 1)
    d["Overwork"] = (d["Weekly_Work_Hours"] > 45).astype(float)
    d["Low_Satisfaction"] = (d["Satisfaction_Score"] < 5).astype(float)
    return d


X_train_fe = add_features(X_train)
NUM_FE = NUM_COLS + ["Income_per_Exp_Year", "Overwork", "Low_Satisfaction"]
fe_rows = []
for name, mdl in [("Logistic Regression", LogisticRegression(max_iter=2000, class_weight="balanced")),
                  ("Random Forest", RandomForestClassifier(n_estimators=200, max_depth=4, min_samples_leaf=5,
                                                           class_weight="balanced", random_state=SEED))]:
    before = cross_validate(pipe(mdl), X_train, y_train, cv=cv_rep, scoring="roc_auc")["test_score"]
    after = cross_validate(pipe(mdl, num_cols=NUM_FE), X_train_fe, y_train, cv=cv_rep, scoring="roc_auc")["test_score"]
    fe_rows.append({"Model": name, "AUC_original_features": before.mean(),
                    "AUC_with_new_features": after.mean(), "Std": after.std()})
fe_df = pd.DataFrame(fe_rows)
print(fe_df.round(3).to_string(index=False))
fe_df.round(4).to_csv("results/feature_engineering_check.csv", index=False)

              Model  AUC_original_features  AUC_with_new_features   Std
Logistic Regression                  0.576                  0.582 0.055
      Random Forest                  0.580                  0.569 0.049


## 11. Hyperparameter tuning (GridSearchCV, scored on ROC-AUC)

GridSearchCV over a small grid per model, scored with ROC-AUC. Plus how much C and tree depth matter.

In [12]:
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
grids = {
    "Logistic Regression": (pipe(LogisticRegression(max_iter=2000, class_weight="balanced")),
                            {"model__C": [0.001, 0.01, 0.1, 1, 10]}),
    "PCA + Logistic Regression": (pipe(LogisticRegression(max_iter=2000, class_weight="balanced"), with_pca=True),
                                  {"pca__n_components": [2, 4, 6, 8, 12],
                                   "model__C": [0.01, 0.1, 1]}),
    "Random Forest": (pipe(RandomForestClassifier(n_estimators=200, class_weight="balanced",
                                                  random_state=SEED)),
                      {"model__max_depth": [2, 3, 4, 6],
                       "model__min_samples_leaf": [2, 5, 10, 20]}),
    "KNN": (pipe(KNeighborsClassifier()),
            {"model__n_neighbors": [5, 11, 21, 31, 41]}),
}
tuned, tune_rows = {}, []
for name, (model, grid) in grids.items():
    gs = GridSearchCV(model, grid, cv=cv5, scoring="roc_auc", n_jobs=-1)
    gs.fit(X_train, y_train)
    tuned[name] = gs.best_estimator_
    tune_rows.append({"Model": name, "Combinations": len(gs.cv_results_["params"]),
                      "Best_params": str(gs.best_params_), "Best_CV_ROC_AUC": gs.best_score_})
    print(f"{name:<28} best CV AUC {gs.best_score_:.3f}  {gs.best_params_}")
tune_df = pd.DataFrame(tune_rows)
tune_df.round(4).to_csv("results/hyperparameter_tuning.csv", index=False)

# how much does one hyperparameter matter? (random forest depth)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
Cs = [0.001, 0.01, 0.1, 1, 10]
cs = []
for C in Cs:
    r = cross_validate(pipe(LogisticRegression(C=C, max_iter=2000, class_weight="balanced")),
                       X_train, y_train, cv=cv5, scoring="roc_auc")
    cs.append(r["test_score"])
ax[0].errorbar(range(len(Cs)), [c.mean() for c in cs], yerr=[c.std() for c in cs], fmt="o-", capsize=4)
ax[0].set_xticks(range(len(Cs))); ax[0].set_xticklabels(Cs)
ax[0].axhline(0.5, color="gray", ls=":")
ax[0].set_xlabel("C (smaller = stronger regularization)"); ax[0].set_ylabel("CV ROC-AUC")
ax[0].set_title("Logistic Regression: effect of C (mean \u00b1 std over folds)")
depths = [2, 3, 4, 6, 10]
ds = []
for d in depths:
    r = cross_validate(pipe(RandomForestClassifier(n_estimators=200, max_depth=d, min_samples_leaf=5,
                                                   class_weight="balanced", random_state=SEED)),
                       X_train, y_train, cv=cv5, scoring="roc_auc")
    ds.append(r["test_score"])
ax[1].errorbar(range(len(depths)), [c.mean() for c in ds], yerr=[c.std() for c in ds], fmt="o-", capsize=4, color="#55A868")
ax[1].set_xticks(range(len(depths))); ax[1].set_xticklabels(depths)
ax[1].axhline(0.5, color="gray", ls=":")
ax[1].set_xlabel("max_depth"); ax[1].set_ylabel("CV ROC-AUC")
ax[1].set_title("Random Forest: effect of depth (mean \u00b1 std over folds)")
plt.tight_layout()
plt.savefig("figures/06_tuning_effect.png", dpi=110)
plt.close()

Logistic Regression          best CV AUC 0.594  {'model__C': 0.01}
PCA + Logistic Regression    best CV AUC 0.594  {'model__C': 1, 'pca__n_components': 4}
Random Forest                best CV AUC 0.592  {'model__max_depth': 2, 'model__min_samples_leaf': 20}
KNN                          best CV AUC 0.584  {'model__n_neighbors': 41}


![06_tuning_effect.png](figures/06_tuning_effect.png)

## 12. Pick the final model and set a decision threshold (train data only)

Pick the model with the best CV ROC-AUC, then choose a decision threshold from out-of-fold predictions on the training data.

In [13]:
best_name = tune_df.loc[tune_df.Best_CV_ROC_AUC.idxmax(), "Model"]
final = tuned[best_name]
print("selected model (highest CV ROC-AUC):", best_name)

oof = cross_val_predict(final, X_train, y_train, cv=cv5, method="predict_proba")[:, 1]
ths = np.linspace(0.2, 0.8, 61)
f1s = [f1_score(y_train, oof >= t) for t in ths]
best_t = float(ths[int(np.argmax(f1s))])
print(f"threshold with the best out-of-fold F1: {best_t:.2f} (F1 {max(f1s):.3f})")

selected model (highest CV ROC-AUC): PCA + Logistic Regression
threshold with the best out-of-fold F1: 0.35 (F1 0.503)


## 13. Final evaluation on the untouched test set

The one and only look at the test set: metrics at the default and tuned thresholds, confusion matrices, bootstrap interval for ROC-AUC, ROC and precision-recall curves.

In [14]:
final.fit(X_train, y_train)
prob = final.predict_proba(X_test)[:, 1]


def report(y_true, prob, t):
    p = (prob >= t).astype(int)
    return {"Threshold": round(t, 2),
            "Accuracy": accuracy_score(y_true, p),
            "Precision": precision_score(y_true, p, zero_division=0),
            "Recall": recall_score(y_true, p, zero_division=0),
            "F1": f1_score(y_true, p, zero_division=0)}


test_rows = [{"Setting": "default threshold 0.50", **report(y_test, prob, 0.5)},
             {"Setting": "tuned threshold", **report(y_test, prob, best_t)}]
test_df = pd.DataFrame(test_rows)
auc = roc_auc_score(y_test, prob)
test_df["ROC_AUC"] = auc
print(test_df.round(3).to_string(index=False))
test_df.round(4).to_csv("results/test_metrics.csv", index=False)

# how sure can we be about that AUC? bootstrap the test set
rng = np.random.RandomState(SEED)
boots = []
yt = y_test.to_numpy()
for _ in range(2000):
    i = rng.randint(0, len(yt), len(yt))
    if yt[i].min() != yt[i].max():
        boots.append(roc_auc_score(yt[i], prob[i]))
lo, hi = np.percentile(boots, [2.5, 97.5])
print(f"\ntest ROC-AUC = {auc:.3f}, 95% bootstrap interval [{lo:.3f}, {hi:.3f}]")
print("baseline test accuracy (always 'stayed'):", round(1 - yt.mean(), 3))

# confusion matrices
fig, ax = plt.subplots(1, 2, figsize=(9, 3.8))
for a, t, title in zip(ax, [0.5, best_t], ["default threshold 0.50", f"tuned threshold {best_t:.2f}"]):
    cm = confusion_matrix(y_test, (prob >= t).astype(int))
    a.imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2):
            a.text(j, i, cm[i, j], ha="center", va="center", fontsize=15,
                   color="white" if cm[i, j] > cm.max() / 2 else "black")
    a.set_xticks([0, 1]); a.set_yticks([0, 1])
    a.set_xticklabels(["stayed", "left"]); a.set_yticklabels(["stayed", "left"])
    a.set_xlabel("predicted"); a.set_ylabel("actual"); a.set_title(title)
    tn, fp, fn, tp = cm.ravel()
    print(f"{title:<26} TN={tn} FP={fp} FN={fn} TP={tp}")
    if t == 0.5:
        pd.DataFrame(cm, index=["Actual 0", "Actual 1"],
                     columns=["Predicted 0", "Predicted 1"]).to_csv("results/confusion_matrix.csv")
plt.tight_layout()
plt.savefig("figures/07_confusion_matrices.png", dpi=110)
plt.close()

# ROC and precision-recall curves for all tuned models
fig, ax = plt.subplots(1, 2, figsize=(11, 4.4))
for name, model in tuned.items():
    if name != best_name:
        model.fit(X_train, y_train)
    pr = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, pr)
    ax[0].plot(fpr, tpr, label=f"{name} (AUC {roc_auc_score(y_test, pr):.2f})")
    p_, r_, _ = precision_recall_curve(y_test, pr)
    ax[1].plot(r_, p_, label=name)
ax[0].plot([0, 1], [0, 1], "k--", label="random guessing")
ax[0].set_xlabel("false positive rate"); ax[0].set_ylabel("true positive rate")
ax[0].set_title("ROC curves (test set)"); ax[0].legend(fontsize=8)
ax[1].axhline(y_test.mean(), color="k", ls="--", label="share of leavers")
ax[1].set_xlabel("recall"); ax[1].set_ylabel("precision")
ax[1].set_title("Precision-recall curves (test set)"); ax[1].legend(fontsize=8)
plt.tight_layout()
plt.savefig("figures/08_roc_pr_curves.png", dpi=110)
plt.close()

               Setting  Threshold  Accuracy  Precision  Recall    F1  ROC_AUC
default threshold 0.50       0.50      0.47      0.339   0.636 0.442    0.536
       tuned threshold       0.35      0.33      0.319   0.909 0.472    0.536

test ROC-AUC = 0.536, 95% bootstrap interval [0.404, 0.668]
baseline test accuracy (always 'stayed'): 0.67
default threshold 0.50     TN=26 FP=41 FN=12 TP=21
tuned threshold 0.35       TN=3 FP=64 FN=3 TP=30


![07_confusion_matrices.png](figures/07_confusion_matrices.png)

![08_roc_pr_curves.png](figures/08_roc_pr_curves.png)

## Takeaways

- **Clusters:** the data splits into a junior group and a senior group (age, experience and income move together, r > 0.85). K = 2 is the clearest and most stable choice, but silhouette is only 0.32, so this is a gradual spectrum cut in two, not two sharply separated groups.
- **Hierarchical:** average linkage has the highest silhouette but only because it splits off 16 outliers; Ward gives balanced groups and agrees reasonably with K-Means (ARI 0.58).
- **Attrition:** the senior cluster has a higher attrition rate (39.7% vs 27.0%, chi-square p = 0.004), found without using the label.
- **Classification:** every model stays near ROC-AUC 0.55–0.59; the test AUC of 0.54 has a 95% interval of 0.40–0.67, so it is not distinguishable from guessing. Tuning, PCA and extra features do not change that.
- **Threshold:** lowering it to 0.35 raises recall to 91% but only by flagging almost everyone, a trade-off and not an improvement.